# Kinetiq Studio — Google Colab GPU Worker (L4 / A100)
Pull-based ComfyUI worker using Colab Secrets (`API_BASE_URL`, `WORKER_TOKEN`, optional `HF_TOKEN`).
Uploads finished 1080p/4K masters and 540p proxies directly to your Google Drive via 8 MiB resumable chunks.

In [ ]:
# Cell 1: Load Secrets & Auto-Tune VRAM Flags
import os, torch
from google.colab import userdata

os.environ['API_BASE_URL'] = userdata.get('API_BASE_URL')
os.environ['WORKER_TOKEN'] = userdata.get('WORKER_TOKEN')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

PROFILE = 'ltx25_pro'  # options: 'ltx25_pro' | 'wan22_fast' | 'upscale_4k'

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9)
comfy_vram_flag = '--highvram' if vram_gb >= 70 else ('--normalvram' if vram_gb >= 38 else '--lowvram')
print(f'GPU: {gpu_name} ({vram_gb} GB) | Flag: {comfy_vram_flag} | Profile: {PROFILE}')

In [ ]:
%%bash
# Cell 2: Install Headless ComfyUI, Custom Nodes & kinetiq-agent-core
set -e
pip install -q hf_transfer huggingface_hub httpx websocket-client psutil

if [ ! -d '/content/ComfyUI' ]; then
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
  pip install -q -r /content/ComfyUI/requirements.txt
fi

cd /content/ComfyUI/custom_nodes
for repo in \
  'https://github.com/Lightricks/ComfyUI-LTXVideo.git' \
  'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git' \
  'https://github.com/1038lab/ComfyUI-FlashVSR.git' \
  'https://github.com/city96/ComfyUI-GGUF.git'; do
  name=$(basename "$repo" .git)
  if [ ! -d "$name" ]; then
    git clone --depth 1 "$repo"
    [ -f "$name/requirements.txt" ] && pip install -q -r "$name/requirements.txt"
  fi
done

In [ ]:
# Cell 3: Fast Profile-Aware Model Bootstrap via hf_transfer
from pathlib import Path
from huggingface_hub import hf_hub_download

MODELS = Path('/content/ComfyUI/models')

def fetch(repo_id: str, filename: str, subdir: str):
    dest = MODELS / subdir
    dest.mkdir(parents=True, exist_ok=True)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(dest))

# Always fetch Qwen-Image FP8 for crisp 1080p keyframes
fetch('Comfy-Org/Qwen-Image', 'split_files/diffusion_models/qwen_image_fp8_e4m3fn.safetensors', 'diffusion_models')
fetch('Comfy-Org/Qwen-Image', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', 'text_encoders')
fetch('Comfy-Org/Qwen-Image', 'split_files/vae/qwen_image_vae.safetensors', 'vae')

if PROFILE == 'wan22_fast':
    base = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
    fetch(base, 'split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors', 'diffusion_models')
    fetch(base, 'split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors', 'diffusion_models')
    fetch(base, 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors', 'loras')
    fetch(base, 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors', 'loras')
    fetch(base, 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders')
    fetch(base, 'split_files/vae/wan_2.1_vae.safetensors', 'vae')
elif PROFILE == 'ltx25_pro':
    fetch('unsloth/LTX-2.5-FP8', 'ltx-2.5-22b-distilled-transformer-fp8.safetensors', 'diffusion_models')
    fetch('Lightricks/LTX-2.5', 'gemma4-12b-with-proj-ltx-2.5-bf16.safetensors', 'text_encoders')
    fetch('Lightricks/LTX-2.5', 'ltx-2.5-video-vae-bf16.safetensors', 'vae')
    fetch('Lightricks/LTX-2.5', 'ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors', 'latent_upscale_models')
print('Model bootstrap complete for profile:', PROFILE)

In [ ]:
# Cell 4: Start Headless ComfyUI Server (127.0.0.1:8188)
import subprocess, time, httpx

comfy_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188',
     comfy_vram_flag, '--disable-auto-launch', '--preview-method', 'none'],
    stdout=open('/content/comfy.log', 'w'),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        if httpx.get('http://127.0.0.1:8188/system_stats', timeout=2).status_code == 200:
            print('Headless ComfyUI ready on 127.0.0.1:8188')
            break
    except Exception:
        time.sleep(2)

In [ ]:
# Cell 5: Launch Pull-Based GPU Worker (install kinetiq-agent-core & runner.py from your repo)
!pip install -q -e /content/kinetiq-studio/workers/agent-core
!python /content/kinetiq-studio/workers/gpu/runner.py --profile $PROFILE --provider google_colab --idle-minutes 15